In [1]:
import numpy as np
from plotly.subplots import make_subplots
import plotly.graph_objects as go

from helpers import *
from qtripy import QTripy
from reader import read_geom_peacs_model
# from readECGsim import read_ecgsim_model

Read data

In [ ]:
# model = read_ecgsim_model('ECGsim_data')


In [2]:
model = read_geom_peacs_model('./PPD1_ECGSIM_NEW','PPD1_ECGSIM_NEW')

a = model.VENTR.THORAX

q = QTripy()
q.begin()

Starting QTriplot...
Connected to QTriplot at 127.0.0.1:1041


In [3]:
vertices, triangles = model.GEOM.ventr.VER, model.GEOM.ventr.ITRI

refPos = model.GEOM.thorax.VER[37, :] - model.GEOM.thorax.NORMV[37, :] * 20
iatrial = 1006; atrialPos = model.ATRIA.geom.VER[iatrial-1,:] + model.ATRIA.geom.NORMV[iatrial-1,:] * 1
irvapex =   39; rvapexPos = model.GEOM.ventr.VER[irvapex-1,:] + model.GEOM.ventr.NORMV[irvapex-1,:] * 0.5
irvcoil =  119; rvcoilPos = model.GEOM.ventr.VER[irvcoil-1,:] + model.GEOM.ventr.NORMV[irvcoil-1,:] * 19
ilvprox =  962; lvproxPos = model.GEOM.ventr.VER[ilvprox-1,:] + model.GEOM.ventr.NORMV[ilvprox-1,:] * 0.5
ilvmid1 =  905; lvmid1Pos = model.GEOM.ventr.VER[ilvmid1-1,:] + model.GEOM.ventr.NORMV[ilvmid1-1,:] * 0.5
ilvmid2 = 1207; lvmid2Pos = model.GEOM.ventr.VER[ilvmid2-1,:] + model.GEOM.ventr.NORMV[ilvmid2-1,:] * 0.5
ilvdist =  636; lvdistPos = model.GEOM.ventr.VER[ilvdist-1,:] + model.GEOM.ventr.NORMV[ilvdist-1,:] * 0.5

pos_points=[rvapexPos,rvcoilPos,lvproxPos,lvmid1Pos,lvmid2Pos,lvdistPos,atrialPos]


compute the transmembrane potentials 

In [4]:
# mark foci
HD = model.VENTR.HEARTDIST
foci = 978
fociact = 0

# get depolarization and repolarization times
dep = HD[foci-1,:] + fociact
rep = 250 + np.mean(dep) - dep*0.6

# transmembrane potentials
maxt = 550
T = np.ones((len(dep), 1)) * np.arange(0, maxt + 1)
p = np.zeros(5)
p[0] = 2
p[1] = 0
p[2] = -0.025
p[3] = -0.02
p[4] = 0
S = gets(T, dep, rep, p, 4)
St = getTris_prof(dep, rep, p, maxt, model.GEOM.ventr.VER, model.GEOM.ventr.ITRI)


In [5]:

q.clear()
q.send_command('bgdcolor grey')

q.send_surface(vertices, triangles)
q.send_values(rep)
q.send_command('cross y')
q.send_surface(vertices, triangles)
q.send_command('trans 0.5')
q.send_values(rep)
q.send_command('funcolor hsv')
q.send_command('mouse ver')

q.send_marker(model.GEOM.ventr.VER[foci,:], 'blue', 5)
# for pos in pos_points:
#     q.send_marker(np.asarray(pos),'red', 3)

electrodes = [90, 72]
fig = make_subplots(rows=1, cols=len(electrodes)+1, subplot_titles=[f"Electrodes in {e} point" for e in electrodes])

for i, e in enumerate(electrodes):
    E1 = model.VENTR.VENTRICLES[e-1, :] @ S
    E2 = zeromean(model.VENTR.tVENTRICLES[e-1, :]) @ St
    x = np.arange(0, len(E1))
    fig.add_trace(go.Scatter(x=x, y=E1, line=dict(color='black', width=1), name='E1'), row=1, col=1+i)
    fig.add_trace(go.Scatter(x=x, y=E2, line=dict(color='red', width=1), name='E2'), row=1, col=1+i)
    q.send_marker(model.GEOM.thorax.VER[e-1,:],'red', 3)

a = model.VENTR.THORAX
O1 = model.VENTR.THORAX[0, :] @ S
O2 = model.VENTR.tTHORAX[0, :] @ St
x = np.arange(0, len(O1))
fig.add_trace(go.Scatter(x=x, y=O1, line=dict(color='black', width=1), name='O1'), row=1, col=len(electrodes)+1)
fig.add_trace(go.Scatter(x=x, y=O2, line=dict(color='red', width=1), name='O2'), row=1, col=len(electrodes)+1)

fig.show()





created
incoming


cylinder

In [11]:
if(False):
    q.clear()
    ver, itri, center_idx = q.send_cylinder(25, nseg=32, axis=(-1, -1, 0.3), radius=3)
    pos0 = ver[center_idx[0]]
    pos1 = ver[center_idx[1]]
    q.send_command('trans 0.3')
    q.send_command('edge y')
    q.send_marker(pos0, 'green', 1, name='bottom')
    q.send_marker(pos1, 'red', 1, name='top')
